In [19]:
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM
import json
import pandas as pd
from tqdm import tqdm

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
print(device)

cuda


In [3]:

tokenizer = LlamaTokenizer.from_pretrained('sarvamai/OpenHathi-7B-Hi-v0.1-Base')
model = LlamaForCausalLM.from_pretrained(
    'sarvamai/OpenHathi-7B-Hi-v0.1-Base',
    torch_dtype=torch.bfloat16
)
model = model.to(device)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
prompt = "australia की राजधानी क्या है? एक छोटे से वाक्य में in hindi"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate
generate_ids = model.generate(inputs.input_ids, max_length=30)
response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print(response)

australia की राजधानी क्या है? एक छोटे से वाक्य में in hindi.
---
ऑस्ट्रेलिया की राजधानी कैनबरा है।


In [37]:
# ...existing code...



# Load test.json
with open("test.json", "r") as f:
    test_data = json.load(f)

# Few-shot template (Hindi, as per your example)
FEW_SHOT = """
# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

def table_to_text(table):
    # Convert a list of dicts to a pipe-separated table string
    if not table:
        return ""
    columns = list(table[0].keys())
    header = " | ".join(columns)
    rows = []
    for i, row in enumerate(table):
        row_str = " | ".join(str(row[col]) for col in columns)
        rows.append(f"<row {i+1}> {row_str}")
    return f"<column>\n{header}\n" + "\n".join(rows)

def resultdf_to_text(result_df):
    # Convert result_df (list of dicts) to a string for display
    if not result_df:
        return ""
    df = pd.DataFrame(result_df)
    return df.to_string(index=False)


In [38]:
for table in test_data:
    table_name = table["table_name"]
    full_table = table["full_table_df"]
    table_text = table_to_text(full_table)
    for q in table["queries"]:
        question = q["question"]
        result_df = q["result_df"]
        # Compose the prompt
        prompt = FEW_SHOT + f"\n###Input:\n{question}\n{table_text}\n\n###Response:\n"
        print(prompt)
        # Tokenize and generate
        # inputs = tokenizer(prompt, return_tensors="pt").to(device)
        # generate_ids = model.generate(inputs.input_ids, max_length=256)
        # response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        # # Print results
        print("="*80)
        # print(f"Question: {question}")
        # print(f"Table: {table_name}")
        # print("\nModel Response:\n", response.split("###Response:")[-1].strip())
        # print("\nActual Result DF:\n", resultdf_to_text(result_df))
        print("="*80)


# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.

###Input:
'नगर पालिका' प्रकार के नगरों का क्षेत्रफल (वर्ग किमी में) क्या है?
<column>
नगर का नाम | प्रकार | तहसील | क्षेत्रफल(वर्ग किमी) | जनसँख्या(२०११) | साक्षरता दर(२०११)
<row 1> पौड़ी | नगर पालिका | पौड़ी | 42.0 | 25440.0 | 90.2
<row 2> काशीरामपुर | जनगणना नगर | कोटद्वार | 19.1 | 10837.0 | 86.4
<row 3> श्रीनगर | नगर पालिका | श्रीनगर | 8.0 | 20115.0 | 90.67
<row 4> लैंसडाउन | छावनी परिषद् | लैंसडाउन | 6.0 | 5687.0 | 94.51
<row

In [ ]:
total_queries = sum(len(table["queries"]) for table in test_data)

with tqdm(total=total_queries, desc="Processing queries") as pbar:
    for table in test_data:
        table_name = table["table_name"]
        full_table = table["full_table_df"]
        table_text = table_to_text(full_table)
        for q in table["queries"]:
            question = q["question"]
            result_df = q["result_df"]
            # Compose the prompt
            prompt = FEW_SHOT + f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
            # Tokenize and generate
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            generate_ids = model.generate(inputs.input_ids)
            response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            # Print results
            print("="*80)
            print(f"Question: {question}")
            print(f"Table: {table_name}")
            print("\nModel Response:\n", response.split("###Response:")[-1].strip())
            print("\nActual Result DF:\n", resultdf_to_text(result_df))
            print("="*80)
            pbar.update(1)
            
        

Processing queries:   2%|▏         | 1/50 [00:14<12:06, 14.83s/it]

Question: 'नगर पालिका' प्रकार के नगरों का क्षेत्रफल (वर्ग किमी में) क्या है?
Table: उत्तराखण्ड के नगरों की सूची_table_11

Model Response:
 <column>
नगर का नाम | प्रकार | तहसील | क्षेत्रफल(वर्ग किमी) | जनसँख्या(२०११) | साक्षरता दर(२०११)
<row 1> पौड़ी | नगर पालिका | पौड़ी | 42.0 | 25440.0 | 90.2
<row 2> काशीरामपुर | जनगणना नगर | कोटद्वार | 19.1 | 10837.0 | 86.4
<row 3> श्रीनगर | नगर पालिका | श्रीनगर | 8.0 | 20115.0 | 90.67
<row 4> लैंसडाउन | छावनी परिषद् | लैंसडाउन | 6.0 | 5687.0 | 94.51
<row 5> दुगड्डा | नगर पंचायत | कोटद्वार | 3.0 | 2422.0 | 89.25
<row 6> बाह बाजार | नगर पंचायत | देवप्रयाग | 1.0 | 716.0 | 96.31
<row 7> कोटद्वार | नगर पालिका | कोटद्वार | 3.0 | 33035.0 | 80.82

संदर्भः
1. https://en.wikipedia.org/wiki/List_of_cities_in_Uttarakhand_by_population
2. https://en.wikipedia.org/wiki/List_of_cities_in_Uttarakhand_by_area
3. https://en.wikipedia.org/wiki/List_of_cities_in_Uttarakhand_by_literacy_rate
4. https://en.wikipedia.org/wiki/List_of_cities_in_Uttarakhand_by_tehsil
5. htt

In [40]:
print(response)


# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.

###Input:
'नगर पालिका' प्रकार के नगरों का क्षेत्रफल (वर्ग किमी में) क्या है?
<column>
नगर का नाम | प्रकार | तहसील | क्षेत्रफल(वर्ग किमी) | जनसँख्या(२०११) | साक्षरता दर(२०११)
<row 1> पौड़ी | नगर पालिका | पौड़ी | 42.0 | 25440.0 | 90.2
<row 2> काशीरामपुर | जनगणना नगर | कोटद्वार | 19.1 | 10837.0 | 86.4
<row 3> श्रीनगर | नगर पालिका | श्रीनगर | 8.0 | 20115.0 | 90.67
<row 4> लैंसडाउन | छावनी परिषद् | लैंसडाउन | 6.0 | 5687.0 | 94.51
<row